PLASTRE Rémi 

ROUGEOT Pauline

SEBIRE Romain

# Expected Utility Analysis - Plastic Classification

### Scope

This notebook implements and analyzes a classifier based on **expected utility** for the problem of sorting plastics. 

The goal is to compare different **classical classification algorithms** with our expected utility approach for sorting **ABS**, **HIPS**, **PE**, and **PP** plastics.

### Imports and configuration

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import BaseEstimator, ClassifierMixin

from scipy.spatial.distance import cdist, pdist

# Graph's configuration
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

# Pandas configuration to plot results
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

### Task 1: Assessing utility

Here is the utility matrix corresponding to the plastic sorting problem. 

\begin{array}{c|cccc}
\text{Act} & \text{ABS} & \text{HIPS} & \text{PE} & \text{PP} \\ \hline
\text{predict ABS} & x_1 & x_2 & x_3 & x_4 \\
\text{predict HIPS} & x_5 & x_6 & x_7 & x_8 \\
\text{predict PE} & x_9 & x_{10} & x_{11} & x_{12} \\
\text{predict PP} & x_{13} & x_{14} & x_{15} & x_{16} \\
\end{array}

### Utility Matrix Construction using Fractile Method

We constructed the utility matrix using the fractile method based on decision maker's indifference between degenerate lotteries and two-outcome lotteries.

### Justification of Fractile Values (π) from Decision Maker Perspective

The fractile values π represent the decision maker's **indifference probability** between:
- A **degenerate lottery** giving consequence x¹ with certainty
- A **two-outcome lottery** [π : x*, (1-π) : x⁰] where x* is the best outcome (+10) and x⁰ is the worst (-10)

**Decision maker's rationale for each π:**

| Consequence | π | Decision Maker's Justification |
|-------------|---|-------------------------------|
| **Correct classification** | 1.0 | *Perfect sorting is the ideal outcome. The certainty of correct classification is prefered over any risky lottery* |
| **ABS ⇔ HIPS confusion** | 0.85 | *These are both styrenic plastics with similar chemical properties. The contamination is manageable. This outcome is acceptable unless there's at least an 85% chance of perfect profit versus 15% risk of major loss."* |
| **PE ⇔ PP confusion** | 0.70 | *Polyolefin mixing degrades material properties, but we can still sell it as lower-grade material. It's problematic but not catastrophic. Acceptable only if there's a 70% chance of maximum profit against 30% risk of major loss."* |
| **Cross-contamination** | 0.15 | *Mixing styrenics with polyolefins causes severe quality issues. Acceptable only it if there's just a 15% chance of profit versus 85% chance of major loss.* |

**Decision-theoretic interpretation:**
- **High π (0.85)**: Perfect
- **Medium π (0.70)**: Acceptable
- **Low π (0.15)**: Terrible

In [4]:
def build_U_4bins():
    # π represents the probability in the equivalent lottery [π : best, (1-π) : worst]

    fractiles = {
        'correct_classification': 1.0,     # Certain best outcome
        'abs_hips_confusion': 0.85,        # Styrenic mix - quite tolerable
        'pe_pp_confusion': 0.70,           # Polyolefin mix - moderately problematic  
        'cross_contamination': 0.15        # Styrenic-polyolefin mix - very bad
    }

    # Realistic utility scale: u(x*) = +10 (profit), u(x0) = -10 (major loss)
    u_max, u_min = 10.0, -10.0

    # Calculate utilities: u(x) = π * u_max + (1-π) * u_min
    utilities = {k: pi * u_max + (1-pi) * u_min for k, pi in fractiles.items()}

    # Build the utility matrix using fractile-derived values with negative scale
    u_correct = utilities['correct_classification']       # 10.0
    u_styrenic = utilities['abs_hips_confusion']         # 7.0  
    u_polyolefin = utilities['pe_pp_confusion']          # 4.0
    u_contamination = utilities['cross_contamination']   # -7.0

    # Construct the 4x4 utility matrix
    U_fractile = np.array([
        # Actions vs States:     ABS        HIPS       PE            PP
        [u_correct,     u_styrenic,  u_contamination, u_contamination],  # Predict ABS
        [u_styrenic,    u_correct,   u_contamination, u_contamination],  # Predict HIPS  
        [u_contamination, u_contamination, u_correct,  u_polyolefin],    # Predict PE
        [u_contamination, u_contamination, u_polyolefin, u_correct]      # Predict PP
    ])

    return U_fractile

In [ ]:
U_fractile = build_U_4bins()

print("Utility matrix for a classification into 4 bins")
print(f"{'State of the world':<15} {'ABS':<8} {'HIPS':<8} {'PE':<8} {'PP':<8}")
for i, action in enumerate(['ABS', 'HIPS', 'PE', 'PP']):
    print(f"{'Predict ' + action:<15} {U_fractile[i,0]:<8.0f} {U_fractile[i,1]:<8.0f} {U_fractile[i,2]:<8.0f} {U_fractile[i,3]:<8.0f}")


Utility matrix for a classification into 4 bins
State of the world ABS      HIPS     PE       PP      


NameError: name 'U_4bins' is not defined

**Utility matrix using fractile method :**

\begin{array}{c|cccc}
\text{Act} & \text{ABS} & \text{HIPS} & \text{PE} & \text{PP} \\ \hline
\text{predict ABS} & +10.0 & +7.0 & -7.0 & -7.0 \\
\text{predict HIPS} & +7.0 & +10.0 & -7.0 & -7.0 \\
\text{predict PE} & -7.0 & -7.0 & +10.0 & +4.0 \\
\text{predict PP} & -7.0 & -7.0 & +4.0 & +10.0 \\
\end{array}

**Fractile justification:**
- **+10.0**: Correct classification (π = 1.0) - Maximum profit from pure material
- **+7.0**: ABS ⇔ HIPS confusion (π = 0.85) - Minor quality loss  
- **+4.0**: PE ⇔ PP confusion (π = 0.70) - Degraded properties
- **-7.0**: Cross-contamination (π = 0.15) - Major loss, batch likely rejected

### Task 2 

**Objectives:**

- Implement the expected utility model.
- Test various classical classification algorithms. 
- Analyse example showing the expected behaviour of the decision maker.

##### 1. Parameters & Utility Fonctions

In [21]:
# Plastic classes

CLASSES = ["ABS", "HiPS", "PE", "PP"]
IDX = {c:i for i,c in enumerate(CLASSES)}

def expected_utility_4bins(P, U):
    """
    Calculates the expected utility for each classification action.

    Parameters:
    - P: (n,4) class probabilities in the order of CLASSES
    - U: (4,4) 4-bin utility matrix

    Returns:
    - EU: expected utilities for each action 
    - best_action_idx: indices of the best actions 
    - best_action: names of the best actions
    - EU_max: maximum expected utilities 
    """
    
    EU = P @ U.T  # (n,4)
    best_action_idx = EU.argmax(axis=1)
    best_action = np.array(CLASSES)[best_action_idx]

    return EU, best_action_idx, best_action, EU.max(axis=1)

##### 2. Data Loading and Analysis

In [23]:
df = pd.read_csv("PlasticsTrain.csv", sep=";", decimal=",")

print(f"Dataset's shape: {df.shape}")
df.head()

Dataset's shape: (10254, 158)


,3687cm-1,3665cm-1,3642cm-1,3620cm-1,3598cm-1,3577cm-1,3555cm-1,3534cm-1,3514cm-1,3493cm-1,3473cm-1,3453cm-1,3433cm-1,3413cm-1,3394cm-1,3375cm-1,3356cm-1,3337cm-1,3318cm-1,3300cm-1,3282cm-1,3264cm-1,3246cm-1,3229cm-1,3211cm-1,3194cm-1,3177cm-1,3160cm-1,3144cm-1,3127cm-1,3111cm-1,3095cm-1,3079cm-1,3063cm-1,3047cm-1,3032cm-1,3017cm-1,3001cm-1,2986cm-1,2972cm-1,2957cm-1,2942cm-1,2928cm-1,2914cm-1,2899cm-1,2885cm-1,2872cm-1,2858cm-1,2844cm-1,2831cm-1,2817cm-1,2804cm-1,2791cm-1,2778cm-1,2765cm-1,2752cm-1,2740cm-1,2727cm-1,2715cm-1,2703cm-1,2690cm-1,2678cm-1,2666cm-1,2655cm-1,2643cm-1,2631cm-1,2620cm-1,2608cm-1,2597cm-1,2586cm-1,2574cm-1,2563cm-1,2552cm-1,2542cm-1,2531cm-1,2520cm-1,2509cm-1,2499cm-1,2489cm-1,2478cm-1,2468cm-1,2458cm-1,2448cm-1,2438cm-1,2428cm-1,2418cm-1,2408cm-1,2399cm-1,2389cm-1,2379cm-1,2370cm-1,2361cm-1,2351cm-1,2342cm-1,2333cm-1,2324cm-1,2315cm-1,2306cm-1,2297cm-1,2288cm-1,2280cm-1,2271cm-1,2262cm-1,2254cm-1,2245cm-1,2237cm-1,2229cm-1,2220cm-1,2212cm-1,2204cm-1,2196cm-1,2188cm-1,2180cm-1,2172cm-1,2164cm-1,2156cm-1,2148cm-1,2141cm-1,2133cm-1,2125cm-1,2118cm-1,2110cm-1,2103cm-1,2096cm-1,2088cm-1,2081cm-1,2074cm-1,2067cm-1,2059cm-1,2052cm-1,2045cm-1,2038cm-1,2031cm-1,2025cm-1,2018cm-1,2011cm-1,2004cm-1,1997cm-1,1991cm-1,1984cm-1,1978cm-1,1971cm-1,1965cm-1,1958cm-1,1952cm-1,1945cm-1,1939cm-1,1933cm-1,1927cm-1,1920cm-1,1914cm-1,1908cm-1,1902cm-1,1896cm-1,class,line,column,object
0,-1.9962,-1.9648,-1.9881,-1.9980,-1.9971,-1.9819,-1.9819,-1.9361,-1.9541,-1.9066,-1.8644,-1.8680,-1.8250,-1.7587,-1.7488,-1.7139,-1.6529,-1.5830,-1.5149,-1.4261,-1.3670,-1.3051,-1.2290,-1.1106,-1.0318,-0.9583,-0.8794,-0.7682,-0.6759,-0.5908,-0.4698,-0.4250,-0.3891,-0.3326,-0.2403,-0.2152,-0.1856,-0.1184,-0.1408,-0.1579,-0.1354,-0.0611,0.0223,0.2016,0.3468,0.4014,0.4454,0.4534,0.6085,0.7259,0.8003,0.8362,0.8541,0.8657,0.9195,0.8899,0.9652,1.0432,1.0369,1.1006,1.0907,1.1705,1.1929,1.2296,1.2243,1.2108,1.2287,1.2010,1.2305,1.1893,1.2028,1.2072,1.2368,1.2431,1.2592,1.2485,1.2583,1.2395,1.2834,1.2476,1.2350,1.2180,1.2386,1.1499,1.1499,1.1454,1.1149,1.1355,1.0558,0.5735,-0.2296,-1.0506,-1.2567,-1.1151,-1.1026,-0.9789,-0.7360,-0.2323,0.1962,0.5431,0.7429,0.7250,0.8003,0.7842,0.7797,0.8371,0.8003,0.7510,0.7205,0.6910,0.6730,0.6551,0.6013,0.5852,0.5735,0.5655,0.5431,0.5108,0.4481,0.4104,0.3647,0.3306,0.2795,0.2016,0.1657,0.1128,0.1039,0.1541,0.1379,0.1612,0.1487,0.0877,0.0626,0.0824,-0.0162,-0.0826,-0.1077,-0.1399,-0.2206,-0.2161,-0.2457,-0.2027,-0.2717,-0.3344,-0.3165,-0.4402,-0.5343,-0.5442,-0.5908,-0.7198,-0.7871,-0.7629,-0.7754,-0.8005,ABS,1,1,ABS_EMA_noir_1_1_22
1,-1.9320,-1.9329,-1.9521,-2.0063,-1.9845,-1.9687,-1.9670,-1.9364,-1.9101,-1.8629,-1.8340,-1.8611,-1.8078,-1.7728,-1.7483,-1.7247,-1.6678,-1.5883,-1.5060,-1.4457,-1.3608,-1.3040,-1.2095,-1.1308,-1.0495,-0.9550,-0.8614,-0.7591,-0.6681,-0.6121,-0.5089,-0.4381,-0.4031,-0.3349,-0.2561,-0.2395,-0.2334,-0.1932,-0.2037,-0.2107,-0.1958,-0.1652,-0.0681,0.1103,0.2678,0.3238,0.3483,0.3815,0.5258,0.6736,0.7541,0.8179,0.8302,0.8923,0.9107,0.9421,0.9754,1.0768,1.0576,1.1346,1.1250,1.1801,1.2273,1.2317,1.2675,1.2588,1.2649,1.2483,1.2605,1.2089,1.2290,1.2168,1.2562,1.2422,1.2544,1.2824,1.2684,1.3086,1.2920,1.2841,1.2693,1.2115,1.2500,1.1547,1.1608,1.1774,1.0847,1.1285,1.0637,0.6054,-0.2037,-1.0468,-1.2480,-1.1054,-1.0705,-0.9944,-0.7407,-0.2360,0.2249,0.5389,0.7226,0.7637,0.7917,0.7690,0.7891,0.8276,0.7978,0.7593,0.7314,0.6911,0.6763,0.6194,0.5704,0.5914,0.5547,0.5407,0.5328,0.4926,0.4663,0.4174,0.3570,0.3264,0.2485,0.2275,0.1628,0.1235,0.1331,0.1383,0.1226,0.1488,0.1515,0.1182,0.0596,0.1051,0.0010,-0.0803,-0.1022,-0.1337,-0.1870,-0.2168,-0.2421,-0.2063,-0.2745,-0.3305,-0.2972,-0.4197,-0.5527,-0.5579,-0.5649,-0.6917,-0.7661,-0.7538,-0.7529,-0.7827,ABS,1,2,ABS_EMA_noir_1_1_22
2,-1.9331,-1.9322,-1.9736,-1.9463,-1.9832,-1.9656,-1.9489,-1.9269,-1.9040,-1.8714,-1.8477,-1.8494,-1.8160,-1.7860,-1.7561,-1.7420,-1.6830,-1.6012,-1.5703,-1.4480,-1.3731,-1.3071,-1.2428,-1.1266,-1.0562,-0.9514,-0.8660,-0.7472,-0.672

In [24]:
# Spectral features extraction
feature_cols = [col for col in df.columns if col not in ['class', 'line', 'column', 'object']]
print(f"Number of spectral features: {len(feature_cols)}")
print(f"Spectral range: {feature_cols[0]} à {feature_cols[-1]}")

# Data processing
X = df[feature_cols].values
y = df['class'].values

print(f"\nData shape: X={X.shape}, y={y.shape}")

Number of spectral features: 154
Spectral range: 3687cm-1 à 1896cm-1

Data shape: X=(10254, 154), y=(10254,)


In [25]:
# Splitting into training and testing set
X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set size: {X_tr.shape[0]} samples")
print(f"Validation set size: {X_va.shape[0]} samples")

# Stratification
train_dist = pd.Series(y_tr).value_counts(normalize=True).sort_index()
val_dist = pd.Series(y_va).value_counts(normalize=True).sort_index()

stratification_df = pd.DataFrame({
    'Train': train_dist,
    'Validation': val_dist
})
print("\nStratification verification:")
print(stratification_df.round(3))

Training set size: 7177 samples
Validation set size: 3077 samples

Stratification verification:
      Train  Validation
ABS   0.191       0.191
HiPS  0.286       0.286
PE    0.285       0.285
PP    0.238       0.238


##### 4. Expected utility classifer implementation

In [26]:
class ExpectedUtilityClassifier(BaseEstimator, ClassifierMixin):
    """    
    Classifier that selects the class/bin based on expected utility.
    
    For 4-bin sorting: Actions = Classes = ["ABS", "HiPS", "PE", "PP"]
    Each action corresponds to sorting into a specific bin.

    Parameters:
    -----------
    base_classifier : estimator
        Base classifier providing P(class|X).
    utility_matrix : array-like, shape (4, 4)
        Utility matrix U[i,j] = utility of classifying as class i when true class is j
    classes : array-like
        List of classes ["ABS", "HiPS", "PE", "PP"]
    """
    
    def __init__(self, base_classifier, utility_matrix, classes=None):
        self.base_classifier = base_classifier
        self.utility_matrix = np.array(utility_matrix)
        self.classes = classes if classes is not None else CLASSES
        self.classes_ = np.array(self.classes)
        
    def fit(self, X, y):
        """Train base classifier"""
        self.base_classifier.fit(X, y)
        return self
    
    def predict_proba(self, X):
        """Return class probabilities"""
        if hasattr(self.base_classifier, "predict_proba"):
            proba = self.base_classifier.predict_proba(X)
            # Reorder according to self.classes
            base_classes = list(self.base_classifier.classes_)
            reorder_idx = [base_classes.index(c) for c in self.classes]
            return proba[:, reorder_idx]
        else:
            raise ValueError("Base classifier must implement predict_proba")
    
    def predict_utility(self, X):
        """
        Compute expected utility for each classification action (= each bin)
        
        Returns:
        --------
        EU : array, shape (n_samples, 4)
            Expected utility for each bin/class [ABS, HiPS, PE, PP]
        """
        P = self.predict_proba(X)       # (n_samples, 4)
        EU = P @ self.utility_matrix.T  # (n_samples, 4)
        return EU
    
    def predict(self, X):
        """        
        Predict the bin/class that maximizes expected utility.

        Returns
        -------
        predictions : array, shape (n_samples,)
            Optimal bin/class labels for each sample
        """
        EU = self.predict_utility(X)  # Shape: (n_samples, 4)
        best_bin_idx = EU.argmax(axis=1)  # Indices in [0,1,2,3]
        return np.array([self.classes[i] for i in best_bin_idx])

##### 5. Classifier comparison function

Test with multiple **classical classification algorithms**

In [27]:
def compare_classifiers(base_classifiers, X_train, y_train, X_val, y_val):
    """
    Compare classical classifiers with EU classifiers for 4 bins
    
    Returns:
    --------
    results : dict
        Comparison results for each classifier
    """
    U = build_U_4bins()
    
    results = {}
    
    for name, base_clf in base_classifiers:
        print(f"\n--- Evaluation: {name} ---")
        
        # Classical classifier
        base_clf.fit(X_train, y_train)
        y_pred_classic = base_clf.predict(X_val)
        acc_classic = accuracy_score(y_val, y_pred_classic)
        
        # Expected Utility (EU) classifier
        eu_clf = ExpectedUtilityClassifier(base_clf, U, CLASSES)
        eu_clf.fit(X_train, y_train)  # Already trained but for consistency
        
        # EU predictions 
        y_pred_eu = eu_clf.predict(X_val)
        acc_eu = accuracy_score(y_val, y_pred_eu)
        
        # EU utilities
        EU = eu_clf.predict_utility(X_val)
        avg_eu = EU.max(axis=1).mean()
        
        # Distribution of EU classifications by class
        eu_distribution = pd.Series(y_pred_eu).value_counts(normalize=True).sort_index()
        
        results[name] = {
            'classic_accuracy': acc_classic,
            'eu_accuracy': acc_eu,
            'avg_expected_utility': avg_eu,
            'eu_distribution': eu_distribution,
            'eu_classifier': eu_clf,
            'classic_predictions': y_pred_classic,
            'eu_predictions': y_pred_eu,
            'utilities': EU
        }
        
        print(f"  Classic accuracy: {acc_classic:.3f}")
        print(f"  EU accuracy: {acc_eu:.3f}")
        print(f"  Average expected utility: {avg_eu:.3f}")
        print(f"  EU distribution: {dict(eu_distribution.round(3))}")
    
    return results

In [28]:
# Classical classifiers definition
base_classifiers = [
    ("LogReg", LogisticRegression(max_iter=500)),
    ("SVM-RBF", SVC(kernel="rbf", probability=True)),
    ("kNN-15", KNeighborsClassifier(n_neighbors=15)),
    ("RF-300", RandomForestClassifier(n_estimators=300, random_state=42)),
    ("DecisionTree", DecisionTreeClassifier(random_state=42, max_depth=10))
]

# Standardization for the models which require it
standardized_classifiers = []
for name, clf in base_classifiers:
    if name in ["LogReg", "SVM-RBF", "kNN-15"]:
        standardized_classifiers.append((name, make_pipeline(StandardScaler(), clf)))
    else:
        standardized_classifiers.append((name, clf))

**Comparison of best EKNN with classical classifiers**

In [29]:
# Classifiers comparison 
print("Classifiers comparison - 4 bins classification")

results = compare_classifiers(standardized_classifiers, X_tr, y_tr, X_va, y_va)

Classifiers comparison - 4 bins classification

--- Evaluation: LogReg ---
  Classic accuracy: 0.918
  EU accuracy: 0.919
  Average expected utility: 8.966
  EU distribution: {'ABS': np.float64(0.195), 'HiPS': np.float64(0.285), 'PE': np.float64(0.29), 'PP': np.float64(0.231)}

--- Evaluation: SVM-RBF ---


Exception ignored on calling ctypes callback function: <function ThreadpoolController._find_libraries_with_dl_iterate_phdr.<locals>.match_library_callback at 0x7f9c79e30220>
Traceback (most recent call last):
  File "/home/default_user/Documents/AdvancedMachineLearning/GraphicalApproachesAndModels/GraphicalApproachesAndModel/assignementvenv/lib/python3.11/site-packages/threadpoolctl.py", line 1005, in match_library_callback
  File "/home/default_user/Documents/AdvancedMachineLearning/GraphicalApproachesAndModels/GraphicalApproachesAndModel/assignementvenv/lib/python3.11/site-packages/threadpoolctl.py", line 1187, in _make_controller_from_path
  File "/home/default_user/Documents/AdvancedMachineLearning/GraphicalApproachesAndModels/GraphicalApproachesAndModel/assignementvenv/lib/python3.11/site-packages/threadpoolctl.py", line 114, in __init__
  File "/usr/lib/python3.11/ctypes/__init__.py", line 376, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^

  Classic accuracy: 0.906
  EU accuracy: 0.912
  Average expected utility: 8.925
  EU distribution: {'ABS': np.float64(0.191), 'HiPS': np.float64(0.305), 'PE': np.float64(0.272), 'PP': np.float64(0.232)}

--- Evaluation: kNN-15 ---
  Classic accuracy: 0.893
  EU accuracy: 0.886
  Average expected utility: 8.372
  EU distribution: {'ABS': np.float64(0.18), 'HiPS': np.float64(0.297), 'PE': np.float64(0.297), 'PP': np.float64(0.226)}

--- Evaluation: RF-300 ---
  Classic accuracy: 0.925
  EU accuracy: 0.918
  Average expected utility: 7.931
  EU distribution: {'ABS': np.float64(0.196), 'HiPS': np.float64(0.305), 'PE': np.float64(0.278), 'PP': np.float64(0.22)}

--- Evaluation: DecisionTree ---
  Classic accuracy: 0.843
  EU accuracy: 0.843
  Average expected utility: 8.922
  EU distribution: {'ABS': np.float64(0.171), 'HiPS': np.float64(0.292), 'PE': np.float64(0.31), 'PP': np.float64(0.228)}


**Results Analysis**

In [30]:
comparison_df = pd.DataFrame([
    {
        'Model': name,
        'Classic_Accuracy': res['classic_accuracy'],
        'EU_Accuracy': res['eu_accuracy'],
        'Accuracy_Diff': res['eu_accuracy'] - res['classic_accuracy'],
        'Avg_Expected_Utility': res['avg_expected_utility'],
    }
    for name, res in results.items()
])

comparison_df = comparison_df.sort_values('Avg_Expected_Utility', ascending=False)

print("Comparative summary: Classic vs Expected utility")
print(comparison_df.round(4))

print(f"\nBest model according to expected utility: {comparison_df.iloc[0]['Model']}")
print(f"Expected utility: {comparison_df.iloc[0]['Avg_Expected_Utility']:.4f}")

Comparative summary: Classic vs Expected utility
          Model  Classic_Accuracy  EU_Accuracy  Accuracy_Diff  \
0        LogReg            0.9181       0.9191         0.0010   
1       SVM-RBF            0.9058       0.9119         0.0062   
4  DecisionTree            0.8427       0.8434         0.0006   
2        kNN-15            0.8928       0.8856        -0.0071   
3        RF-300            0.9249       0.9181        -0.0068   

   Avg_Expected_Utility  
0                8.9664  
1                8.9248  
4                8.9216  
2                8.3723  
3                7.9311  

Best model according to expected utility: LogReg
Expected utility: 8.9664


##### Examples illustrating expected behaviour

In [32]:
def analyze_decision_examples(results, X_val, y_val, n_examples=10):
    """
    Analyze examples where the EU classifier makes different decisions 
    from the classical classifier, illustrating the expected behavior 
    of the decision maker.
    """    
    # Take the best model according to expected utility
    best_model = max(results.items(), key=lambda x: x[1]['avg_expected_utility'])
    model_name, model_results = best_model
    
    print(f"\nAnalysis based on model: {model_name}")
    print(f"Average expected utility: {model_results['avg_expected_utility']:.3f}")
    
    eu_clf = model_results['eu_classifier']
    classic_pred = model_results['classic_predictions']
    eu_pred = model_results['eu_predictions']
    utilities = model_results['utilities']
    
    # Calculate probabilities
    P = eu_clf.predict_proba(X_val)
    
    # Create DataFrame for analysis
    analysis_df = pd.DataFrame({
        'y_true': y_val,
        'classic_pred': classic_pred,
        'eu_pred': eu_pred,
        'P_ABS': P[:, 0],
        'P_HiPS': P[:, 1], 
        'P_PE': P[:, 2],
        'P_PP': P[:, 3],
        'EU_ABS': utilities[:, 0],
        'EU_HiPS': utilities[:, 1],
        'EU_PE': utilities[:, 2],
        'EU_PP': utilities[:, 3],
        'max_EU': utilities.max(axis=1),
        'confidence': P.max(axis=1)
    })
    
    return analysis_df, model_name

# Corrected examples analysis
analysis_df, best_model_name = analyze_decision_examples(results, X_va, y_va, n_examples=5)


Analysis based on model: LogReg
Average expected utility: 8.966


In [34]:
# Analysis of different behaviors between EU and classical classifier
print("\nAnalysis of expected utility vs classic decisions")
print("Examples where EU chooses differently from classical classifier")

# Identify samples where EU differs from classical classifier
different_decisions = analysis_df['classic_pred'] != analysis_df['eu_pred']

if different_decisions.sum() > 0:
    print(f"Differences found: {different_decisions.sum()}/{len(analysis_df)} ({different_decisions.mean():.1%})")
    
    examples = analysis_df[different_decisions].head(5)
    
    for idx, (_, row) in enumerate(examples.iterrows()):
        print(f"\nExample {idx+1}:")
        print(f"  True class: {row['y_true']}")
        print(f"  Classic: {row['classic_pred']} | EU: {row['eu_pred']}")
        print(f"  Probabilities: ABS={row['P_ABS']:.3f}, HiPS={row['P_HiPS']:.3f}, PE={row['P_PE']:.3f}, PP={row['P_PP']:.3f}")
        print(f"  EU utilities: ABS={row['EU_ABS']:.3f}, HiPS={row['EU_HiPS']:.3f}, PE={row['EU_PE']:.3f}, PP={row['EU_PP']:.3f}")
        
        # Analyze why EU chose differently
        best_eu_idx = np.argmax([row['EU_ABS'], row['EU_HiPS'], row['EU_PE'], row['EU_PP']])
        best_prob_idx = np.argmax([row['P_ABS'], row['P_HiPS'], row['P_PE'], row['P_PP']])
        
        if best_eu_idx != best_prob_idx:
            print(f"  → EU avoids risk: chooses {CLASSES[best_eu_idx]} instead of {CLASSES[best_prob_idx]} (more probable)")
        else:
            print(f"  → Decision consistent with probabilities")
else:
    print("  No differences found between EU and classical classifier.")


Analysis of expected utility vs classic decisions
Examples where EU chooses differently from classical classifier
Differences found: 20/3077 (0.6%)

Example 1:
  True class: PE
  Classic: PP | EU: HiPS
  Probabilities: ABS=0.224, HiPS=0.267, PE=0.210, PP=0.300
  EU utilities: ABS=0.538, HiPS=0.667, PE=-0.136, PP=0.402
  → EU avoids risk: chooses HiPS instead of PP (more probable)

Example 2:
  True class: PP
  Classic: PE | EU: ABS
  Probabilities: ABS=0.316, HiPS=0.193, PE=0.323, PP=0.167
  EU utilities: ABS=1.082, HiPS=0.714, PE=0.334, PP=-0.602
  → EU avoids risk: chooses ABS instead of PE (more probable)

Example 3:
  True class: HiPS
  Classic: HiPS | EU: PP
  Probabilities: ABS=0.086, HiPS=0.368, PE=0.208, PP=0.338
  EU utilities: ABS=-0.396, HiPS=0.450, PE=0.263, PP=1.042
  → EU avoids risk: chooses PP instead of HiPS (more probable)

Example 4:
  True class: HiPS
  Classic: PE | EU: HiPS
  Probabilities: ABS=0.267, HiPS=0.292, PE=0.321, PP=0.121
  EU utilities: ABS=1.621, HiPS

##### Summary of Task 2

**Tests with multiple classical algorithms**
- 5 algorithms tested: LogReg, SVM-RBF, kNN-15, RF-300, GBDT
- Logistic Regression identified as best model (EU = 0.361)

**Decision maker behavior analysis**
- Prudent rejection: 99% for PE/PP (avoids cost -5.0)
- Strategic sending: 85-87% for ABS/HiPS (aims for gain +1.0)

**Key points**
- Theoretical consistency: EU classifier respects principles of decision under uncertainty
- Rational behavior: Decisions aligned with decision maker's preferences
- Robustness: Consistent results across different base algorithms

### Task 3: Evidential k-Nearest Neighbourhood (EKNN) Classifier

**Objectives** 

Same objectives as in Task 2 using a EKNN (Evidential k-Nearest Neighbourhood) classifier. 

**EKNN concept**

1. Each neighbor provides a *belief mass* based on its distance.
2. It *quantifies uncertainty and ignorance* explicitly.
3. Neighbor evidences are *combined using Dempster’s rule*.
4. *Pignistic probabilities* convert belief masses into decision-ready probabilities.

This method is used for *classification under high uncertainty*.


##### EKNN Implementation

In [40]:
class EKNNClassifier:
    """
    Evidential k-NN (EKNN):
      - per-neighbor mass m_j({θ_l}|x)=φ_l(d_j), m_j(Θ|x)=1-φ_l(d_j)
      - combine neighbors of same class (products), then combine across classes (Dempster)
      - betP = m({θ_l}) + m(Θ)/n
    Decision (predict) = argmax betP by default.
    """
    def __init__(self, k=30, alpha=0.99, gammas=None, metric='euclidean', n_jobs=None):
        self.k = int(k)
        self.alpha = float(alpha)
        self.metric = metric
        self.gammas_user = gammas  # dict {class_label: gamma} or None
        self.n_jobs = n_jobs

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        self.X_ = X
        self.y_ = y
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        self.nn_ = NearestNeighbors(n_neighbors=self.k, metric=self.metric, n_jobs=self.n_jobs)
        self.nn_.fit(X)

        # Estimate gamma_l per class if not provided
        if self.gammas_user is not None:
            self.gammas_ = {c: float(self.gammas_user.get(c, 1.0)) for c in self.classes_}
        else:
            self.gammas_ = {}
            for c in self.classes_:
                Xc = X[y == c]
                gamma = 1.0
                if Xc.shape[0] >= 3:
                    # average pairwise distance within class (robust to scaling with small epsilon)
                    D = pdist(Xc, metric=self.metric)
                    avg = np.mean(D) if len(D) else 1.0
                    gamma = 1.0 / max(avg, 1e-8)
                else:
                    # fallback: use global average distance
                    Dg = pdist(X, metric=self.metric)
                    avg = np.mean(Dg) if len(Dg) else 1.0
                    gamma = 1.0 / max(avg, 1e-8)
                self.gammas_[c] = gamma
        return self

    def _phi(self, d, c):
        # φ_l(d) = α * exp(-γ_l d^2), clipped to [0, 1]
        val = self.alpha * np.exp(- self.gammas_[c] * (d ** 2))
        return np.clip(val, 0.0, 1.0)

    def _combine_masses(self, dists, neigh_labels):
        """
        Combine neighbor masses:
          Step 1 (per class): for class ℓ, m_l(θ_l)=1 - prod_j (1-φ_l(d_j)), m_l(Θ)=prod_j (1-φ_l(d_j))
          Step 2 (all classes, Dempster): m({θ_l}) = m_l(θ_l) * ∏_{r≠l} m_r(Θ) / Z
                                         m(Θ)     = ∏_l m_l(Θ) / Z
          Z = sum_l m_l(θ_l)*∏_{r≠l} m_r(Θ) + ∏_l m_l(Θ)
        Returns m_singletons (size n_classes) and m_Theta (scalar).
        """
        m_theta_l = np.zeros(self.n_classes_, dtype=float)
        m_Theta_l = np.ones(self.n_classes_, dtype=float)
        # For each class, gather distances of neighbors of that class
        for idx_c, c in enumerate(self.classes_):
            mask = (neigh_labels == c)
            dj = dists[mask]
            if dj.size == 0:
                # No neighbor of this class among the k: m_l(θ_l)=0, m_l(Θ)=1
                m_theta_l[idx_c] = 0.0
                m_Theta_l[idx_c] = 1.0
            else:
                one_minus_phi = 1.0 - self._phi(dj, c)
                prod = np.prod(one_minus_phi) if one_minus_phi.size else 1.0
                m_theta_l[idx_c] = 1.0 - prod
                m_Theta_l[idx_c] = prod

        # Combine across classes with normalizing factor
        # numerator per class l: m_l(θ_l) * prod_{r!=l} m_r(Θ)
        numer_singletons = np.zeros_like(m_theta_l)
        for l in range(self.n_classes_):
            others = np.delete(m_Theta_l, l)
            numer_singletons[l] = m_theta_l[l] * (others.prod() if others.size else 1.0)
        numer_theta = numer_singletons.sum()
        numer_Theta = m_Theta_l.prod() if m_Theta_l.size else 1.0

        Z = numer_theta + numer_Theta
        if Z <= 0:
            # extreme conflict: fall back to vacuous mass on Θ
            m_singletons = np.zeros_like(m_theta_l)
            m_Theta = 1.0
        else:
            m_singletons = numer_singletons / Z
            m_Theta = numer_Theta / Z
        return m_singletons, m_Theta

    def predict_betp(self, X):
        """
        Compute pignistic probabilities for each sample.
        betP(θ_l) = m({θ_l}) + m(Θ)/n (only singletons and Θ are focal after combination).
        """
        X = np.asarray(X)
        dists, idxs = self.nn_.kneighbors(X, n_neighbors=self.k, return_distance=True)
        betP = np.zeros((X.shape[0], self.n_classes_), dtype=float)
        for i in range(X.shape[0]):
            neigh_idx = idxs[i]
            neigh_labels = self.y_[neigh_idx]
            dd = dists[i]
            m_singletons, m_Theta = self._combine_masses(dd, neigh_labels)
            betP[i] = m_singletons + (m_Theta / self.n_classes_)
            # numerical clean
            s = betP[i].sum()
            if s > 0:
                betP[i] /= s
        return betP

    def predict(self, X):
        betP = self.predict_betp(X)
        arg = np.argmax(betP, axis=1)
        return self.classes_[arg]

    def predict_expected_utility(self, X, U, actions_are_classes=True):
        """
        Expected-utility decision based on betP.
        Parameters
        U : array-like shape (A, C) where C = n_classes (states), A = #actions.
            If actions_are_classes=True, A == C and action a predicts class classes_[a].
        Returns
        preds : chosen action indices (or mapped class labels if actions_are_classes).
        EU    : per-sample expected utilities for all actions.
        """
        betP = self.predict_betp(X)  # shape (N, C)
        U = np.asarray(U, dtype=float)  # (A, C)
        # EU for each sample: EU[a] = sum_c U[a,c]*betP[c]
        EU = betP @ U.T  # shape (N, A)
        a_idx = np.argmax(EU, axis=1)
        if actions_are_classes and U.shape[0] == self.n_classes_:
            return self.classes_[a_idx], EU
        return a_idx, EU

##### Comparison with classical k-NN

In [38]:
def compare_knn_vs_eknn(X_train, y_train, X_val, y_val, k=30, alpha=0.99, gammas=None, U=None):
    """
    Trains classic kNN and EKNN; evaluates accuracy; if U provided, also EU-based EKNN decision.
    Returns dict with metrics and diff_examples indices where knn and eknn differ on the predictions.
    """
    # Classic kNN
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn.fit(X_train, y_train)
    y_pred_knn = knn.predict(X_val)
    acc_knn = accuracy_score(y_val, y_pred_knn)

    # EKNN (MAP via betP argmax)
    eknn = EKNNClassifier(k=k, alpha=alpha, gammas=gammas).fit(X_train, y_train)
    y_pred_eknn_map = eknn.predict(X_val)
    acc_eknn_map = accuracy_score(y_val, y_pred_eknn_map)

    # EKNN expected-utility decision (if U provided)
    eu = None
    y_pred_eknn_eu = None
    acc_eknn_eu = None
    if U is not None:
        y_pred_eknn_eu, EU = eknn.predict_expected_utility(X_val, U, actions_are_classes=True)
        acc_eknn_eu = accuracy_score(y_val, y_pred_eknn_eu)
        eu = {"EU_matrix": EU}

    # Where classic ≠ EKNN (MAP)
    diff_idx = np.where(y_pred_knn != y_pred_eknn_map)[0]

    # Build a small showcase of disagreements (with betP and neighbor sketch)
    showcase = []
    if diff_idx.size:
        # take up to 5 examples
        for i in diff_idx[:5]:
            betp = eknn.predict_betp(X_val[i:i+1])[0]
            # nearest neighbors labels for context
            dists, idxs = eknn.nn_.kneighbors(X_val[i:i+1], n_neighbors=eknn.k, return_distance=True)
            neigh_labels = eknn.y_[idxs[0]]
            showcase.append({
                "val_index": int(i),
                "true": y_val[i],
                "knn_pred": y_pred_knn[i],
                "eknn_pred_map": y_pred_eknn_map[i],
                "betP": {str(cls): float(b) for cls, b in zip(eknn.classes_, betp)},
                "neighbors_labels": neigh_labels.tolist()
            })

    out = {
        "acc_knn": acc_knn,
        "acc_eknn_map": acc_eknn_map,
        "y_pred_knn": y_pred_knn,
        "y_pred_eknn_map": y_pred_eknn_map,
        "diff_examples_idx": diff_idx.tolist(),
        "diff_showcase": showcase,
        "eknn_model": eknn
    }
    if eu is not None:
        out.update({
            "acc_eknn_eu": acc_eknn_eu,
            "y_pred_eknn_eu": y_pred_eknn_eu,
            "EU_details": eu
        })
    return out

##### Comparison between k-NN and EKNN classifiers depending on k

In [ ]:
k_values = [5, 10, 15, 20, 25, 30]
alpha = 0.99

# Calculate automatic gammas once
def calculate_intraclass_distances(X, y, classes):
    """Calculate average intra-class distances"""
    gammas_auto = {}
    for c in classes:
        X_c = X[y == c]
        if X_c.shape[0] >= 2:
            distances = pdist(X_c, metric='euclidean')
            avg_dist = np.mean(distances)
            gammas_auto[c] = 1.0 / max(avg_dist, 1e-8)
        else:
            gammas_auto[c] = 1.0
    return gammas_auto

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_va_scaled = scaler.transform(X_va)

gammas_auto = calculate_intraclass_distances(X_tr_scaled, y_tr, CLASSES)
print(f"Automatic gammas used:")
for cls, gamma in gammas_auto.items():
    print(f"  {cls}: {gamma:.6f}")

print(f"\nTest with alpha={alpha} and gammas=Auto")

results_k = []
for k in k_values:
    print(f"\nTest with k={k}")
    result = compare_knn_vs_eknn(
        X_tr_scaled, y_tr, 
        X_va_scaled, y_va, 
        k=k, 
        alpha=alpha,
        gammas=gammas_auto,
        U=U_fractile
    )
    
    avg_eu = result['EU_details']['EU_matrix'].max(axis=1).mean()
    
    # Calculate error rates
    error_rate_knn = 1 - result['acc_knn']
    error_rate_eknn_map = 1 - result['acc_eknn_map']
    error_rate_eknn_eu = 1 - result['acc_eknn_eu']
    
    results_k.append({
        'k': k,
        'acc_knn': result['acc_knn'],
        'acc_eknn_map': result['acc_eknn_map'],
        'acc_eknn_eu': result['acc_eknn_eu'],
        'error_rate_knn': error_rate_knn,
        'error_rate_eknn_map': error_rate_eknn_map,
        'error_rate_eknn_eu': error_rate_eknn_eu,
        'avg_expected_utility': avg_eu,
        'nb_differences': len(result['diff_examples_idx'])
    })
    
    print(f"  k-NN - Accuracy: {result['acc_knn']:.4f}, Error Rate: {error_rate_knn:.4f}")
    print(f"  E-KNN (MAP) - Accuracy: {result['acc_eknn_map']:.4f}, Error Rate: {error_rate_eknn_map:.4f}")
    print(f"  E-KNN (EU) - Accuracy: {result['acc_eknn_eu']:.4f}, Error Rate: {error_rate_eknn_eu:.4f}")
    print(f"  Expected utility: {avg_eu:.4f}")

# Results analysis
df_k = pd.DataFrame(results_k)
print("Results for different k values:")
print(df_k.round(4))

# Detailed comparisons
print(f"\nAccuracy comparison:")
print(f"{'k':<5} {'k-NN':<8} {'E-KNN(MAP)':<12} {'E-KNN(EU)':<12} {'Diff MAP':<10} {'Diff EU':<10}")
print("-" * 65)
for _, row in df_k.iterrows():
    diff_map = row['acc_eknn_map'] - row['acc_knn']
    diff_eu = row['acc_eknn_eu'] - row['acc_knn']
    print(f"{row['k']:<5.0f} {row['acc_knn']:<8.4f} {row['acc_eknn_map']:<12.4f} {row['acc_eknn_eu']:<12.4f} {diff_map:<10.4f} {diff_eu:<10.4f}")

print(f"\nError rate comparison:")
print(f"{'k':<5} {'k-NN':<8} {'E-KNN(MAP)':<12} {'E-KNN(EU)':<12} {'Diff MAP':<10} {'Diff EU':<10}")
print("-" * 65)
for _, row in df_k.iterrows():
    diff_err_map = row['error_rate_eknn_map'] - row['error_rate_knn']
    diff_err_eu = row['error_rate_eknn_eu'] - row['error_rate_knn']
    print(f"{row['k']:<5.0f} {row['error_rate_knn']:<8.4f} {row['error_rate_eknn_map']:<12.4f} {row['error_rate_eknn_eu']:<12.4f} {diff_err_map:<10.4f} {diff_err_eu:<10.4f}")

# Identification of best parameters
best_k_accuracy = df_k.loc[df_k['acc_eknn_eu'].idxmax()]
best_k_utility = df_k.loc[df_k['avg_expected_utility'].idxmax()]

print("Best parameters:")
print(f"Best k for E-KNN (EU) accuracy: k={best_k_accuracy['k']:.0f}")
print(f"  Accuracy: {best_k_accuracy['acc_eknn_eu']:.4f}")
print(f"  Error Rate: {best_k_accuracy['error_rate_eknn_eu']:.4f}")

print(f"\nBest k for expected utility: k={best_k_utility['k']:.0f}")
print(f"  Expected utility: {best_k_utility['avg_expected_utility']:.4f}")
print(f"  Accuracy: {best_k_utility['acc_eknn_eu']:.4f}")

plt.figure(figsize=(10, 6))

plt.plot(df_k['k'], df_k['acc_knn'], 'o-', label='Classic k-NN', linewidth=2, markersize=8)
plt.plot(df_k['k'], df_k['acc_eknn_map'], 's-', label='E-KNN (MAP)', linewidth=2, markersize=8)
plt.plot(df_k['k'], df_k['acc_eknn_eu'], '^-', label='E-KNN (EU)', linewidth=2, markersize=8)

plt.xlabel('k value')
plt.ylabel('Accuracy')
plt.title('Accuracy Comparison: k-NN vs E-KNN')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(k_values)

# Add values on points
for i, row in df_k.iterrows():
    plt.annotate(f'{row["acc_knn"]:.3f}', 
                (row['k'], row['acc_knn']), 
                textcoords="offset points", xytext=(0,10), ha='center', fontsize=8)
    plt.annotate(f'{row["acc_eknn_eu"]:.3f}', 
                (row['k'], row['acc_eknn_eu']), 
                textcoords="offset points", xytext=(0,-15), ha='center', fontsize=8)

plt.tight_layout()
plt.show()

print("Main observations:")
print(f"1. k-NN accuracy range: {df_k['acc_knn'].min():.4f} - {df_k['acc_knn'].max():.4f}")
print(f"2. E-KNN (EU) accuracy range: {df_k['acc_eknn_eu'].min():.4f} - {df_k['acc_eknn_eu'].max():.4f}")
print(f"3. Expected utility range: {df_k['avg_expected_utility'].min():.4f} - {df_k['avg_expected_utility'].max():.4f}")

diff_acc_eu = df_k['acc_eknn_eu'] - df_k['acc_knn']
print(f"4. E-KNN improves accuracy vs k-NN: {(diff_acc_eu > 0).sum()}/{len(diff_acc_eu)} cases")
print(f"5. Average number of differences between k-NN and E-KNN: {df_k['nb_differences'].mean():.1f}")

: 